# Handwritten Digit Recognition (MNIST CNN)
### End-to-End Production Deep Learning Pipeline in TensorFlow & Keras 3

This notebook demonstrates the end-to-end machine learning lifecycle for handwritten digit recognition on the MNIST dataset using modern Convolutional Neural Networks (CNNs).

In [ ]:
import sys
from pathlib import Path

# Ensure src module is accessible
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

from src.config import DEFAULT_MODEL_CONFIG, DEFAULT_TRAINING_CONFIG, DEFAULT_PATHS
from src.data_loader import load_mnist_raw, get_dataset_summary, create_validation_split
from src.preprocessing import preprocess_pipeline, preprocess_single_image
from src.model import build_cnn_model, build_baseline_model, compile_model, count_parameters
from src.train import train_model
from src.evaluate import evaluate_model
from src.predict import DigitPredictor
from src.utils import set_seed, get_device_info

set_seed(42)
print("System Device Info:", get_device_info())

## 1. Load & Inspect MNIST Dataset

In [ ]:
(x_train_raw, y_train), (x_test_raw, y_test) = load_mnist_raw()
summary = get_dataset_summary(x_train_raw, y_train, x_test_raw, y_test)
print(f"Training set shape: {x_train_raw.shape}, Labels: {y_train.shape}")
print(f"Testing set shape: {x_test_raw.shape}, Labels: {y_test.shape}")
print(f"Pixel value range: [{summary['pixel_min']}, {summary['pixel_max']}]")

## 2. Visualize Sample Digits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5), dpi=120)
axes = axes.flatten()
for i in range(10):
    axes[i].imshow(x_train_raw[i], cmap="gray")
    axes[i].set_title(f"Label: {y_train[i]}", fontweight="bold")
    axes[i].axis("off")
plt.suptitle("Sample MNIST Training Images", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Preprocessing & Tensor Normalization

In [ ]:
x_train_norm = preprocess_pipeline(x_train_raw)
x_test_norm = preprocess_pipeline(x_test_raw)

print(f"Preprocessed train shape: {x_train_norm.shape} | dtype: {x_train_norm.dtype}")
print(f"Preprocessed range: [{x_train_norm.min()}, {x_train_norm.max()}]")

## 4. Build & Inspect CNN Architecture

In [ ]:
model = build_cnn_model(DEFAULT_MODEL_CONFIG)
model = compile_model(model)
model.summary()
print("Parameter Count:", count_parameters(model))

## 5. Model Training with Callbacks
*(EarlyStopping, ModelCheckpoint, ReduceLROnPlateau)*

In [ ]:
trained_model, history, test_metrics = train_model(epochs=10, batch_size=64)
print(f"Test Accuracy: {test_metrics['test_accuracy'] * 100:.2f}%")

## 6. Comprehensive Evaluation & Confusion Matrix

In [ ]:
eval_report = evaluate_model(model=trained_model)
print(f"Overall Test Accuracy : {eval_report['test_accuracy'] * 100:.2f}%")
print(f"Macro F1-Score        : {eval_report['macro_f1']:.4f}")
print(f"Total Misclassified   : {eval_report['total_misclassified']} / {eval_report['test_samples']}")

## 7. Inference Engine on Single Digits

In [ ]:
predictor = DigitPredictor(model=trained_model)
# Test on 1st test sample
result = predictor.predict(x_test_raw[0])
print(f"Predicted Digit: {result['predicted_digit']} with {result['confidence_percent']} confidence")
print("Top candidates:", result["top_k"])